In [19]:
import numpy as np
import nrrd
import argparse
from dipy.io.streamline import load_tractogram,save_tractogram
from dipy.io.utils import Space

#parser = argparse.ArgumentParser()
#parser.add_argument('-t','--tractogram_path', type=str,required=True,help="path of input tractogram")
#parser.add_argument('-o','--output_path_tractogram',type=str,required=True,help="output path of obtained augmented tractogram in .csv")
#parser.add_argument('-r','--reference_path',type=str,required=True,help="reference image or .trk")
#parser.add_argument('-c','--mcm_path',type=str,required=True,help="path of the mcm images")
#parser.add_argument('-w','--weights_path',type=str,required=True,help="weights image")
#parser.add_argument('-s','--coordinate_path',type=str,required=True,help="image of coordinate")

#args = parser.parse_args()
#tractogram_path=args.tractogram_path
#output_path_tractogram=args.output_path_tractogram
#reference=args.reference_path
#mcm_path=args.mcm_path
#weights_path=args.weights_path
#coordinate_path=args.coordinate_path

dataset='PPMI/PD/'
subject='100268/'
tract='CST_left'
path='/home/gui/Documents/Post-doc/data/'

tractogram_path=path+dataset+subject+'05-Tracts/5downsampled_'+tract+'_n40_c30.vtk'
output_path_tractogram=path+dataset+subject+'06-AugmentedTractsOT/6augmented_'+tract+'_n40_c30.trx'
reference=path+dataset+'/000000/02-TransfoToAverageSpace/averageDTI.nii.gz'
mcm_path=path+dataset+subject+'04-AlignedMCM/MCM_avg_aligned/MCM_avg_aligned_'
weights_path=path+dataset+subject+'04-AlignedMCM/MCM_avg_aligned/MCM_avg_aligned_weights.nrrd'
coordinate_path=path+dataset+subject+'04-AlignedMCM/spatial.nrrd'

import sys
sys.path.append('../script/')
from utils import array_to_cov_neighbor, cov_to_array_MAS,most_colinear_compartment
from projection_GMM import proj_GMM_MAS
from distances import distance_confidence_along

d=3

#Compute neigbor kernel
step=np.array([0,1,-1])
step_i, step_j, step_k = np.meshgrid(step, step, step, indexing='ij')
kernel_neighbor = np.stack([step_i, step_j, step_k], axis=-1)[None]


#Load coordinate of mcm image
mcm_coordinate,header= nrrd.read(coordinate_path)
origin_img=header['space origin']
step_img=np.diag(header['space directions'][1:])

#Load weights of compartements and value of iso compartment
mcm_w,_=nrrd.read(weights_path)    
mcm_I,_=nrrd.read(mcm_path+'0.nrrd')
ksize=mcm_w.shape[0]-1 #shape of the array aka maximum number of compartment
    
#Load aniso compartment
mcm_cov=[]
for i in range(ksize):
    mcm_cov+=[nrrd.read(mcm_path+str(i+1)+'.nrrd')[0]]

    
#Load tract
tract = load_tractogram(
            tractogram_path, 
            reference=reference,            
            bbox_valid_check=False,
            trk_header_check=False)#,to_space=Space.LPSMM)
origin_tract=tract.space_attributes[0][:3,-1]
step_tract=np.diag(tract.space_attributes[0])[:3]

nb_streamline=len(tract.streamlines)
            
list_m,list_wI,list_I,list_w,list_S,list_conf,list_col=[],[],[],[],[],[],[]

            
print(nb_streamline,end='->',flush=True)
#for si in range(0,nb_streamline): #nb of streamlines
for si in range(0,1):
    #if si%500==0:
    print(" ",si,end=' ',flush=True)
    streamline_coordinate=tract.streamlines[si] #Get coordinate of the streamline
    nb_pts=streamline_coordinate.shape[0]
    
    #Compute closest voxel from tract coordinate then compute its neigbord
    idx=np.int32(np.round((streamline_coordinate-origin_tract)/step_tract))
    #if np.all((streamline_coordinate[0]*step_tract-mcm_coordinate[:,idx[0,0],idx[0,1],idx[0,2]]*step_img)>1.25):
    #    print("mauvais voxel",Flush=True)
    idx_neighbor=idx[:,None,None,None,:]+kernel_neighbor
        
    #Compute distance to each neigbor
    voxels_neighbor=mcm_coordinate[:,idx_neighbor[:,:,:,:,0],idx_neighbor[:,:,:,:,1],idx_neighbor[:,:,:,:,2]].transpose(1,0,2,3,4)
    D=np.sum((voxels_neighbor-streamline_coordinate[:,:,None,None,None])**2,1)
    
    #Get weights of compartements and value of iso compartment 
    I=mcm_I[idx_neighbor[:,:,:,:,0],idx_neighbor[:,:,:,:,1],idx_neighbor[:,:,:,:,2]]
    wI=mcm_w[0,idx_neighbor[:,:,:,:,0],idx_neighbor[:,:,:,:,1],idx_neighbor[:,:,:,:,2]]
    w=mcm_w[1:,idx_neighbor[:,:,:,:,0],idx_neighbor[:,:,:,:,1],idx_neighbor[:,:,:,:,2]].transpose(1,2,3,4,0)
    kmax=np.max(np.count_nonzero(w,axis=-1),axis=(1,2,3)) #Number of non null compartment  

    #Compute weights of neigbor wrt to distance
    w_dist=1/D
    mask=np.ones((nb_pts,3,3,3))*(np.sum(w,-1)+wI>(1-1e-7)) #Mask for neighbor voxels out of the image    
    w_dist=w_dist*mask
    w_dist/=w_dist.sum((1,2,3))[:,None,None,None]
            
    # Interpolation of iso
    If=np.sum(I*w_dist,(1,2,3))
    wIf=np.sum(wI*w_dist,(1,2,3))
    
    #Get aniso compartment
    S=np.zeros((nb_pts,3,3,3,ksize,d,d))
    for i in range(0,ksize):
        S[:,:,:,:,i]=array_to_cov_neighbor(mcm_cov[i][:,idx_neighbor[:,:,:,:,0],idx_neighbor[:,:,:,:,1],idx_neighbor[:,:,:,:,2]].transpose(1,2,3,4,0))
    
    #Interpolation of aniso
    w=w*w_dist[:,:,:,:,None]
    wf,Sf=proj_GMM_MAS(kmax,ksize,w.reshape(nb_pts,-1),S.reshape(nb_pts,-1,d,d),max_itr=50,eps=1e-7)

    #Compute distance target source
    D=distance_confidence_along(w.reshape(nb_pts,-1),wf,S.reshape(nb_pts,-1,d,d),Sf)
    conf=1/(D+1e-8)
    
    #Compute most colinear compartment
    idx_col=most_colinear_compartment(streamline_coordinate,Sf)
    
    list_m+=[streamline_coordinate]
    list_wI+=[wIf]
    list_I+=[If]
    list_w+=[wf]
    list_S+=[cov_to_array_MAS(Sf)]
    list_conf+=[conf]
    list_col+=[idx_col]

#Normalize confidence 
max_confidence=0
for i in range(len(list_conf)):
    if max(list_conf[i])>max_confidence:
        max_confidence=max(list_conf[i])
for i in range(len(list_conf)):
    list_conf[i]/=max_confidence

tract.streamlines=list_m
dico={}
dico['weight iso']=list_wI
dico['iso']=list_I
dico['weights aniso']=list_w
dico['aniso']=list_S
dico['confidence']=list_conf
dico['colinear']=list_col
tract.data_per_point=dico
#print(tract.data_per_point['weight iso'])
#save_tractogram(tract,'/home/gui/Bureau/6augmented_CST_right_n40_c302.trx')

[2026-06-05 14:19:59][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
1171->  0 

/tmp/ipykernel_20206/2977315421.py:107: RuntimeWarning: invalid value encountered in divide
  w_dist/=w_dist.sum((1,2,3))[:,None,None,None]


ValueError: attempt to get argmin of an empty sequence

In [18]:
list_w

[array([[0.12870754, 0.26538886, 0.28561793, 0.15236983],
        [0.19121304, 0.29014312, 0.08307479, 0.19894149],
        [0.33345172, 0.13951419, 0.24909405, 0.12165539],
        [0.10613244, 0.17797523, 0.59128061, 0.        ],
        [0.09244145, 0.25368441, 0.53923954, 0.        ],
        [0.28681105, 0.41470437, 0.19359219, 0.        ],
        [0.40947707, 0.23854438, 0.25699441, 0.        ],
        [0.22407894, 0.22792811, 0.44320512, 0.        ],
        [0.35289382, 0.21827269, 0.30784118, 0.        ],
        [0.51212218, 0.1926656 , 0.16205466, 0.        ],
        [0.        , 0.22807234, 0.62592594, 0.        ],
        [0.22600732, 0.60568789, 0.        , 0.        ],
        [0.22400955, 0.59932538, 0.        , 0.        ],
        [0.23835263, 0.57163177, 0.        , 0.        ],
        [0.26761715, 0.52897611, 0.        , 0.        ],
        [0.18304465, 0.2868243 , 0.31901607, 0.        ],
        [0.41077837, 0.38144087, 0.03823436, 0.        ],
        [0.363

In [6]:
streamline_coordinate[-1]

array([ 4.88789576, 20.61206422, 55.25323738])

In [7]:
streamline_coordinate[0]

array([18.12015928,  6.75888603, -2.31578696])

In [76]:
#Load tract
tract0 = load_tractogram(
            "/home/gui/Bureau/CST_right.trk",reference='same',to_space=Space.LPSMM)
print(tract0.streamlines[0][0])

[ -0.92168808  25.64941025 -40.9744854 ]


In [100]:


tract = load_tractogram(
            "/home/gui/Bureau/1.vtk",reference=reference,to_space=Space.LPSMM)
print(tract.streamlines[0][0])


save_tractogram(tract,'/home/gui/Bureau/2.vtk')#,to_space=Space.LPSMM)
tract = load_tractogram(
            "/home/gui/Bureau/2.vtk",reference=reference,to_space=Space.LPSMM)
print(tract.streamlines[0][0])

save_tractogram(tract,'/home/gui/Bureau/3.vtk')#,to_space=Space.LPSMM)
tract = load_tractogram(
            "/home/gui/Bureau/3.vtk",reference=reference,to_space=Space.LPSMM)
print(tract.streamlines[0][0])

save_tractogram(tract,'/home/gui/Bureau/4.vtk')#,to_space=Space.LPSMM)
tract = load_tractogram(
            "/home/gui/Bureau/4.vtk",reference=reference,to_space=Space.LPSMM)
print(tract.streamlines[0][0])

[2026-05-21 11:29:34][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[ -3.3621928   34.38545092 -49.66124249]
[2026-05-21 11:29:35][dipy] WARNING: StatefulTractogram was previously saving  in LPSMM space.
Now use to_space=Space.LPSMM to match the previous behavior.
[2026-05-21 11:29:35][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[ -3.3621929   34.38544432 -49.66124778]
[2026-05-21 11:29:35][dipy] WARNING: StatefulTractogram was previously saving  in LPSMM space.
Now use to_space=Space.LPSMM to match the previous behavior.
[2026-05-21 11:29:35][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[ -3.362193    34.38543771 -49.66125308]
[2026-05-21 11:29:35][dipy] WARNING: StatefulTractogram was previously saving  in LPSMM space.
Now use to_space=Space.LPSMM to match the pr

In [ ]:
print("downsampled")
tract1 = load_tractogram(
            "/home/gui/Bureau/5downsampled_CST_right_n40_c30.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])

print("downsampled")
tract1 = load_tractogram(
            "/home/gui/Bureau/5downsampled_CST_right_n40_c30.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])

print("downsampled")
tract1 = load_tractogram(
            "/home/gui/Bureau/5downsampled_CST_right_n40_c30.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])

In [90]:
#Load tract
#Load tract
print("raw")
tract1 = load_tractogram(
            "/home/gui/Bureau/1raw_CST_right.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])
#Load tract
print("aligned")
tract1 = load_tractogram(
            "/home/gui/Bureau/2aligned_CST_right.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])
#Load tract
print("ordered")
tract1 = load_tractogram(
            "/home/gui/Bureau/3reordered_CST_right.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])
print("newgrid")
tract1 = load_tractogram(
            "/home/gui/Bureau/4newgrid_CST_right_n40.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])

print("downsampled")
tract1 = load_tractogram(
            "/home/gui/Bureau/5downsampled_CST_right_n40_c30.vtk",reference=reference)#,to_space=Space.LPSMM)
print(tract1.streamlines[0][0])


raw
[2026-05-21 11:22:50][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[ -0.92168811  25.64941101 -40.97449043]
aligned
[2026-05-21 11:22:50][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[ -0.43937041  25.12619756 -37.84030012]
ordered
[2026-05-21 11:22:50][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[  0.43937042 -25.12619831 -37.84030506]
newgrid
[2026-05-21 11:22:50][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[ -0.43937044  25.12619906 -37.84031   ]
downsampled
[2026-05-21 11:22:50][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[  3.3621927  -34.38545753 -49.66123719]


In [ ]:
#Load tract
tract1 = load_tractogram(
            "/home/gui/Bureau/CST_right.vtk",reference=reference)
print(tract1.streamlines[1][0])

In [50]:
#Load tract
tract2 = load_tractogram(
            "/home/gui/Bureau/6augmented_CST_right_n40_c30.trx", 
            reference=reference)
print(tract2.streamlines[0][0])

[  7.07857073 -29.19187322 -41.38062017]


In [ ]:
from dipy.io.utils import Space

save_tractogram(tract0,'/home/gui/Bureau/CST_right_raw.vtk',to_space=Space.LPSMM)
save_tractogram(tract0,'/home/gui/Bureau/CST_right_raw2.vtk',to_space=Space.LPSMM)

In [22]:
tract = load_tractogram(
            "/home/gui/Bureau/CST_right_raw.vtk",reference=reference)
print(tract.streamlines[0][0])
tract = load_tractogram(
            "/home/gui/Bureau/CST_right_raw2.vtk", from_space=Space.LPSMM, reference=reference)
tract.streamlines[0][0]

[2026-05-21 11:01:16][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.
[ -0.92168811  25.64941101 -40.97449043]
[2026-05-21 11:01:16][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.


array([  0.92168812, -25.64939995, -40.97449295])

In [19]:
#Load tract
tract1 = load_tractogram(
            "/home/gui/Bureau/52downsampled_CST_right_n40_c30.vtk", 
            reference=reference)
tract1.streamlines[0][0]

[2026-05-21 10:58:26][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.


array([ -7.07857063,  29.19188422, -41.38061764])

In [103]:
#Load tract
tract2 = load_tractogram(
            "/home/gui/Bureau/6augmented_CST_right_n40_c30.trx", 
            reference=reference)#,to_space=Space.LPSMM)
print(tract2.streamlines[0][0])

#Load tract
tract2 = load_tractogram(
            "/home/gui/Bureau/6augmented_CST_right_n40_c302.trx", 
            reference=reference)#,to_space=Space.LPSMM)
tract2.streamlines[0][0]

[  7.07857073 -29.19187322 -41.38062017]


array([  7.07857159, -29.1919404 , -41.3806066 ])

In [12]:
tract = load_tractogram(
            tractogram_path, 
            reference=reference)
tract.streamlines[0][0]

[2026-05-21 10:55:13][dipy] WARNING: StatefulTractogram was previously saving in LPSMM space.
Use from_space=Space.LPSMM to load older files.


array([  3.3621927 , -34.38545753, -49.66123719])

In [3]:
T=S.reshape(nb_pts,-1,d,d)

In [5]:
np.linalg.norm(T-T.transpose(0,1,3,2))

np.float64(0.0)

In [19]:
mcm_cov[0].shape

(6, 145, 174, 145)